# RAG Inquiry

### Install and import libraries

In [ ]:
!pip install openai
!pip install chromadb

In [ ]:
from dotenv import load_dotenv
import os
from typing import List
import chromadb
from openai import AzureOpenAI

### Setup OpenAI client

In [ ]:
load_dotenv()

API_Key = os.getenv("AZURE_OPENAI_API_KEY")

if not API_Key:
    raise RuntimeError("Missing Azure OpenAI credentials. Set AZURE_OPENAI_API_KEY in .env or environment.")

client = AzureOpenAI(
    azure_endpoint="https://api-iw.azure-api.net/sig-shared-jpeast/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview",
    api_key=API_Key,
    api_version="2025-01-01-preview",
)

### Setup Chroma DB

In [ ]:
CHROMA_PATH = os.getenv("CHROMA_PATH") or "chroma_db/chroma_db"
COLLECTION_NAME = "rag"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

### Define the question

In [ ]:
question = "What are the current SIGs in InnoWings?"

### Perform retrieval

In [ ]:
top_k = 5

if not question.strip():
    results = collection.peek(limit=top_k)
    context_docs = [{"text": doc, "url": meta.get("url", ""), "score": 1.0}
            for doc, meta in zip(results["documents"], results["metadatas"])]

response = client.embeddings.create(
    input=question.replace("\n", " "),
    model=os.getenv("EMBEDDING_MODEL", "text-embedding-3-small"),
)
query_embedding = response.data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=top_k,
    include=["documents", "metadatas", "distances"]
)

context_docs = []
for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    # Convert distance to similarity score (Chroma uses cosine distance by default)
    score = 1 - distance
    context_docs.append({
        "text": doc,
        "url": meta.get("url", ""),
        "source": meta.get("source", "HKU InnoWings / InnoAcademy"),
        "score": round(score, 4)
    })

### Generate answer

In [ ]:
context_parts = []
for i, doc in enumerate(context_docs, 1):
        context_parts.append(
            f"Document {i} (Source: {doc['url']}):\n{doc['text']}"
        )

context = "\n\n---\n\n".join(context_parts)

messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant for HKU InnoWings and InnoAcademy. "
                "Answer the question using ONLY the provided context documents. "
                "If the answer cannot be found in the context, say: "
                "'I don't have enough information based on the provided documents.' "
                "Be concise, accurate, and professional. Always cite the source URL when possible."
            )
        },
        {
            "role": "user",
            "content": f"Context documents:\n{context}\n\nQuestion: {question}"
        }
    ]

answer = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.3,
    )